In [2]:
import pandas as pd
print("Pandas version:", pd.__version__)

Pandas version: 3.0.5


In [ ]:
import pandas as pd
file = "job_summary.csv"
df_sample = pd.read_csv(file, nrows=5)

print(df_sample.columns.tolist())

['job_link', 'job_summary']


# Preliminary Data Exploration: Business Analyst Job Titles

In [5]:
import pandas as pd
file = "linkedin_job_postings.csv"
df_sample = pd.read_csv(file, nrows=5)

print(df_sample.columns.tolist())

['job_link', 'last_processed_time', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type']


## 1. Dataset Overview

In [6]:
titles = pd.read_csv(
    file,
    usecols=["job_title"]
)

titles.head()

,job_title
0,Account Executive - Dispensing (NorCal/Norther...
1,Registered Nurse - RN Care Manager
2,RESTAURANT SUPERVISOR - THE FORKLIFT
3,Independent Real Estate Agent
4,Group/Unit Supervisor (Systems Support Manager...


## 2. Initial Identification of Business Analyst Job Postings

In [7]:
ba_titles = titles[
    titles["job_title"].str.contains(
        "business analyst",
        case=False,
        na=False
    )
]

print("BA-related postings:", len(ba_titles))

ba_titles["job_title"].value_counts().head(100)

BA-related postings: 4576


job_title
Business Analyst                                                  1011
Senior Business Analyst                                            211
JDE Business Analyst (Supply Chain)                                163
JD Edwards Business Analyst                                        150
IT Business Analyst                                                 64
                                                                  ... 
IT Business Analyst 2                                                3
Business Analyst (Finance)                                           3
Sr. Business Analyst, Quality Improvement - Quality Management       3
Business Analyst - Finance                                           3
Sr. Business Analyst - Anti-Money Laundering                         3
Name: count, Length: 100, dtype: int64

## 3. Broader Identification of Business Analyst-Related Job Postings

In [8]:
ba_broad = titles[
    titles["job_title"].str.contains("business", case=False, na=False)
    & titles["job_title"].str.contains("analyst", case=False, na=False)
]

print("Broad BA-related postings:", len(ba_broad))
print("Unique BA-related titles:", ba_broad["job_title"].nunique())

ba_broad["job_title"].value_counts().head(100)

Broad BA-related postings: 6694
Unique BA-related titles: 3171


job_title
Business Analyst                                   1011
Senior Business Analyst                             211
Business Systems Analyst                            179
JDE Business Analyst (Supply Chain)                 163
JD Edwards Business Analyst                         150
                                                   ... 
Business Analyst - #LI-PG1 #Hiring #Omaha #Jobs       5
Junior Business Analyst                               5
Business Operations Analyst- Mid-Level                5
Business Data Analyst II                              5
Agile Business Analyst                                5
Name: count, Length: 100, dtype: int64

## 4. Exploration of Job Level and Specialisation Keywords

In [9]:
level_keywords = [
    "junior", "jr", "associate",
    "senior", "sr", "lead", "principal"
]

specialisation_keywords = [
    "data", "systems", "system", "IT", "technical",
    "process", "finance", "financial", "supply chain",
    "digital"
]

for word in level_keywords:
    count = ba_broad["job_title"].str.contains(
        rf"\b{word}\b", case=False, na=False, regex=True
    ).sum()
    print(f"{word}: {count}")

print("\n--- Specialisation ---")

for word in specialisation_keywords:
    count = ba_broad["job_title"].str.contains(
        rf"\b{word}\b", case=False, na=False, regex=True
    ).sum()
    print(f"{word}: {count}")

junior: 27
jr: 5
associate: 35
senior: 884
sr: 364
lead: 111
principal: 43

--- Specialisation ---
data: 284
systems: 847
system: 182
IT: 362
technical: 155
process: 140
finance: 89
financial: 137
supply chain: 199
digital: 78


## 5. Comparison of Job Level and Specialisation Coverage

In [10]:
import re

# Level patterns
level_pattern = r'\b(junior|jr|associate|senior|sr|lead|principal|entry|mid[\s-]?level)\b'

# Potential specialisation patterns
spec_pattern = (
    r'\b(data|systems?|IT|technical|process|finance|financial|'
    r'supply chain|digital|operations?|ERP|JDE|JD Edwards)\b'
)

has_level = ba_broad["job_title"].str.contains(
    level_pattern, case=False, na=False, regex=True
)

has_spec = ba_broad["job_title"].str.contains(
    spec_pattern, case=False, na=False, regex=True
)

print("Total BA-related postings:", len(ba_broad))

print("\n--- Job Level ---")
print("Postings with identifiable level:", has_level.sum())
print("Percentage:", round(has_level.mean() * 100, 2), "%")

print("\n--- Specialisation ---")
print("Postings with identifiable specialisation:", has_spec.sum())
print("Percentage:", round(has_spec.mean() * 100, 2), "%")

Total BA-related postings: 6694

--- Job Level ---
Postings with identifiable level: 1462
Percentage: 21.84 %

--- Specialisation ---
Postings with identifiable specialisation: 2566
Percentage: 38.33 %


/var/folders/bm/4f936v8n11l335mmlcgjds9c0000gn/T/ipykernel_21392/2729304116.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_level = ba_broad["job_title"].str.contains(
/var/folders/bm/4f936v8n11l335mmlcgjds9c0000gn/T/ipykernel_21392/2729304116.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_spec = ba_broad["job_title"].str.contains(


## 6. Preliminary Specialisation Classification

In [11]:
spec_groups = {
    "Systems/IT": r'\b(systems?|IT|technical)\b',
    "Data": r'\b(data|analytics?)\b',
    "Finance": r'\b(finance|financial)\b',
    "Process/Operations": r'\b(process|operations?)\b',
    "Supply Chain/ERP": r'\b(supply chain|ERP|JDE|JD Edwards)\b',
    "Digital": r'\bdigital\b'
}

for group, pattern in spec_groups.items():
    count = ba_broad["job_title"].str.contains(
        pattern,
        case=False,
        na=False,
        regex=True
    ).sum()
    
    print(f"{group}: {count}")

Systems/IT: 1445
Data: 332
Finance: 220
Process/Operations: 271
Supply Chain/ERP: 453
Digital: 78


/var/folders/bm/4f936v8n11l335mmlcgjds9c0000gn/T/ipykernel_21392/2861432311.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  count = ba_broad["job_title"].str.contains(


## 7. Inspection of Job Titles within Specialisation Groups

In [12]:
for group, pattern in spec_groups.items():
    print(f"\n===== {group} =====")
    
    matches = ba_broad[
        ba_broad["job_title"].str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )
    ]
    
    print(matches["job_title"].value_counts().head(15))


===== Systems/IT =====
job_title
Business Systems Analyst                                179
IT Business Analyst                                      64
Business System Analyst                                  62
Technical Business Analyst                               60
Senior Business Systems Analyst                          41
IT Business Systems Analyst                              18
Internal Controls Analyst (Business and IT)              18
Business Systems Analyst - Digital Experience            16
Sr. Business Systems Analyst                             15
Senior Business Systems Analyst - Digital Experience     14
Senior Business System Analyst                           14
Senior IT Business Analyst                               11
Business Systems Analyst II                              10
Business Systems Analyst - Sr                             8
Junior Business Systems Analyst                           8
Name: count, dtype: int64

===== Data =====
job_title
Business Dat

/var/folders/bm/4f936v8n11l335mmlcgjds9c0000gn/T/ipykernel_21392/387186523.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ba_broad["job_title"].str.contains(
